# Tools

Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:

1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute.

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [3]:
from langchain.chat_models import init_chat_model
model = init_chat_model('groq:qwen/qwen3-32b')
response = model.invoke('How are you today?')
response

AIMessage(content='<think>\nLet me think about how to engage with this question... I should consider the social dynamics of conversation. When someone asks "How are you today?", they\'re often opening a dialogue, creating a space for connection.\n\nI need to be mindful of the conversational flow. This question might be a prelude to more meaningful discussions. I should be prepared to transition smoothly to other topics they might want to explore.\n\nThe query is deceptively simple, yet it carries social weight. It\'s important to acknowledge the gesture while being honest about my nature as an AI. I should make it clear while still being warm and approachable.\n\nI should consider the emotional tone - it\'s a friendly greeting that often sets the tenor for the conversation. I want to respond in a way that\'s both authentic to my nature and supportive of continued dialogue.\n\nThe question also presents an opportunity to demonstrate my communication abilities. I can show empathy and und

In [5]:
from langchain.tools import tool

@tool
def get_weather(location:str) -> str:
    '''Get weather of a location'''
    return f"It's sunny in {location}."

model_with_tool = model.bind_tools([get_weather])

In [10]:
response = model_with_tool.invoke("How is the weather in Boston?")
print(response)

for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'Okay, the user is asking about the weather in Boston. Let me check the tools available. There\'s a function called get_weather that takes a location parameter. Since the user specified Boston, I need to call that function with "Boston" as the location. I\'ll make sure the parameters are correctly formatted in JSON. Alright, that should retrieve the weather information for them.\n', 'tool_calls': [{'id': 'ns6wwcqd4', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 99, 'prompt_tokens': 152, 'total_tokens': 251, 'completion_time': 0.161314216, 'completion_tokens_details': {'reasoning_tokens': 75}, 'prompt_time': 0.012642327, 'prompt_tokens_details': None, 'queue_time': 0.159804833, 'total_time': 0.173956543}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_2bfcc54d36', 'service_tier': 'on_demand', 'finish_reason': 'tool_call

### Tool Execution Loop

In [11]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tool.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tool.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72°F and sunny."

The weather in Boston is sunny right now. A perfect day to enjoy outdoor activities! 🌞
